## Section A — Setup

* **Consolidated pip install:** `torch`, `torchvision`, `timm`, `albumentations`, `einops`
* **Path Definitions:** Mount Google Drive and define paths:
  * `DRIVE_ROOT`
  * `DATA_ROOT`
  * `OOD_ROOT`
* **Data Extraction & Restoration:**
  * Extract `resized_256` from parts (matching the preprocessing notebook).
  * Restore `backgrounds.zip` from Drive to `/content/dataset/backgrounds/`.
* **Configuration Loading:** Load `class_to_idx.json` and `loss_weights.json`.
* **Directory Rebuild:** Rebuild the soybean flat folder (equivalent to Section C2 — acts as a permanent resume step).
* **Sanity Check:** A single smoke-test cell confirming GPU availability, all paths, and all files are present before touching any training code.

---

## Section B — Dataset Class

* **`CropDataset` Routing:** Reads from manifest CSVs and routes each image to the correct transform based on the `bg_type` column:
  * `paste_eligible` $\rightarrow$ `paste_eligible_transform` (includes `RandomBackgroundPaste`)
  * `natural` / `tight_crop` / anything else $\rightarrow$ `natural_transform`
  * `val`/`test` always $\rightarrow$ `val_transform`
* **Error Handling:** Handles missing files gracefully by logging a warning and skipping the sample rather than crashing mid-epoch.
* **Return Values:** `__getitem__` returns `(image_tensor, label_idx, filepath)`.
  > *Note: The `filepath` is explicitly needed for downstream OOD diagnostics.*

---

## Section C — DataLoaders

* **Train Loader:** `shuffle=True`, `num_workers=2`, `pin_memory=True`, `drop_last=True`
* **Val/Test Loaders:** `shuffle=False`, `num_workers=2`, `pin_memory=True`
* **Imbalance Handling:** No `WeightedRandomSampler` is used. Standard shuffling is applied; class imbalance is handled strictly via loss weighting.
* **Sanity Check:** Quick cell to print one batch's class distribution to confirm sampling looks reasonable.

---

## Section D — Model: DINOv2 ViT-S/14 with Staged Fine-Tuning

* **Backbone:** Load pretrained `dinov2_vits14` via `torch.hub`.
* **Classification Head:** Replace/add `nn.Linear(384, 32)` (ViT-S hidden dimension is 384).
* **Staged Fine-Tuning Strategy:** Controlled via a single `set_trainable_stage(model, stage)` function to easily transition without restarting.

### Fine-Tuning Stages
1. **Stage A (Epochs 1–5): Linear Probe**
   * Backbone is fully frozen; only the head trains.
   * **Learning Rate (LR):** $1\times 10^{-3}$
   * *Purpose:* Establishes baseline OOD accuracy with zero fine-tuning cost (critical diagnostic).
2. **Stage B (Epochs 6–15): Partial Unfreeze**
   * Unfreeze the last 4 transformer blocks + the classification head.
   * **Differential LR:** Backbone LR = $1\times 10^{-5}$, Head LR = $1\times 10^{-4}$
   * *Purpose:* This is where core domain adaptation occurs.
3. **Stage C (Optional, Epochs 16–25): Full Unfreeze**
   * Only triggered if Stage B OOD accuracy plateaus.
   * **LR:** Full model LR = $5\times 10^{-6}$

---

## Section E — Loss, Optimizer, and Scheduler

* **Loss Function:** `nn.CrossEntropyLoss(weight=loss_weights)` loaded from Drive.
* **Regularization:** Label smoothing of `0.1` to help generalization on minority classes.
* **Optimizer:** `AdamW` with weight decay of $1\times 10^{-4}$ and stage-specific differential learning rates.
* **Scheduler:** `CosineAnnealingLR` applied per stage, resetting completely between stage transitions.

---

## Section F — Training Loop

* **Per-Epoch Metrics:** Track train loss, val loss, macro F1, and per-class F1.
* **Checkpointing:** Saves to Drive every epoch **only if** validation macro F1 improves.
  * *Checkpoint contents:* Model state, optimizer state, epoch, best F1, and `class_to_idx`.
* **UX:** `tqdm` progress bar wrapper per batch.
* **Early Stopping:** Patience of 5 epochs **per stage** (does not carry over across stages).
* **Transition Logic:** Auto-advance from Stage A $\rightarrow$ B after 5 epochs; advance from B $\rightarrow$ C only if explicitly triggered.

---

## Section G — Per-Class F1 Monitoring

* **Logging Table:** After every epoch, print the full per-class F1 table sorted in ascending order.
* **Alerting:** Explicitly flag any class falling below a `0.5` F1 floor as an early warning for optimization collapse.
* **Persistence:** Save per-epoch per-class F1 history to Drive as `f1_history.json` to preserve metrics across restarts.
* **Specific Watch List:** Logged every epoch:
  * `['wheat', 'snake_gourd', 'onion', 'dragon_fruit', 'jute', 'blackgram', 'soybean', 'cotton', 'tea']`
  * *(The 7 below-floor classes + cotton/tea to monitor historical collapse/dominance).*

---

## Section H — OOD Evaluation

* **Frequency:** Runs exclusively after every **stage transition** (not every epoch).
* **Data Source:** Loads images from `/content/drive/MyDrive/disease_detection/unseen/Diseased_and_healthy/<crop>/`.
* **Transforms:** No augmentation; uses standard `val_transform` (center crop + normalization).
* **Metrics Reported:** Per-class accuracy on the OOD set, macro accuracy, and a confusion matrix highlighting the worst performing pairs.
* **Artifacts:** Saves OOD results to Drive as `ood_results.json` to allow direct comparison between Stages A, B, and C.
  * *Purpose:* This is the primary metric confirming if DINOv2 + background pasting resolved the field-image collapse.

---

## Section I — Confusion Matrix & Error Analysis

* **Evaluation:** Generate a full confusion matrix on the validation set after each stage.
* **Target Diagnostics:** Highlight the top-5 most confused pairs, focusing specifically on `cotton` $\leftrightarrow$ `tea` confusions and any below-floor crop being misclassified as a visually similar large class.
* **Artifacts:** Save plots to Drive as `confusion_matrix_stage{A/B/C}.png`.

---

## Section J — Resume Capability

* **Structure:** A single execution cell placed at the top of the notebook.
* **Logic:** Detects existing checkpoints on Drive, restores states, and resumes training from the correct epoch and stage.
* **Persistence:** The active stage configuration is saved within the checkpoint metadata so the script handles freezing/unfreezing automatically upon resume.

---

## Execution Expectations Per Stage

* **Stage A (Linear Probe):** Expect a macro F1 of `0.70`–`0.85` on validation, and OOD accuracy around `0.40`–`0.60`. If OOD is $>0.50$ here, it validates that the DINOv2 architecture change is inherently outperforming the previous EfficientNet baseline.
* **Stage B (Partial Unfreeze):** Expect validation macro F1 to climb toward `0.90`–`0.93`. OOD accuracy should show meaningful gains as domain adaptation occurs.
* **Stage C (Full Unfreeze):** Expect marginal gains on validation F1 with possible minor OOD improvements, balanced against a higher risk of overfitting on edge classes like `wheat` and `snake_gourd`. Use selectively.

---
---
---

# SECTION A: Setup — 30-class DINOv2 training run
### Excludes cotton/soybean (see TODO_cotton_soybean_excluded.txt)

In [ ]:
# Install (run once, then RESTART SESSION before continuing)
# !pip install -q torch torchvision timm albumentations einops

import torch, json, shutil, os
from pathlib import Path

# FIX: Mount Drive FIRST before creating any directories
if os.path.exists('/content/drive') and not os.path.ismount('/content/drive'):
    # Clean up the local folder mistakenly created by previous runs
    shutil.rmtree('/content/drive')

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/CropClassifier_v2')
DATA_ROOT  = Path('/content/dataset/resized_256')
OOD_ROOT   = Path('/content/drive/MyDrive/disease_detection/unseen/Diseased_and_healthy')
CKPT_DIR   = DRIVE_ROOT / 'models' / 'stage1_dinov2_30class'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 238   # 17 * 14 — DINOv2 ViT-S/14 patch-divisible

# Restore raw image data (ephemeral, rebuilt every session)
if not DATA_ROOT.exists() or sum(1 for p in DATA_ROOT.iterdir() if p.is_dir()) < 10:
    print("Extracting resized_256 from Drive parts...")
    DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
    import subprocess
    subprocess.run(
        "cat /content/drive/MyDrive/disease_detection/resized_part_* > /content/resized_256.zip",
        shell=True, check=True
    )
    subprocess.run(
        'unzip -q -o /content/resized_256.zip -d /content/dataset -x "__MACOSX/*" "*/._*"',
        shell=True, check=True
    )
    print("✅ Image data extracted")
else:
    print("✅ Image data already on disk, skipping extraction")

# --- Restore background pool (Places365 agricultural crops) ---
BG_POOL_DIR = Path('/content/dataset/backgrounds')
BG_BACKUP_ZIP = DRIVE_ROOT / 'backgrounds.zip'
if not BG_POOL_DIR.exists() or len(list(BG_POOL_DIR.glob('*'))) == 0:
    print("Restoring backgrounds from Drive...")
    import subprocess
    subprocess.run(f"unzip -q -o {BG_BACKUP_ZIP} -d /content/dataset/", shell=True, check=True)
    print(f"✅ Restored {len(list(BG_POOL_DIR.glob('*'))):,} background images")
else:
    print(f"✅ Backgrounds already present: {len(list(BG_POOL_DIR.glob('*'))):,}")

# --- Load 30-class config (NOTE: _30 files, not the original 32-class ones) ---
with open(DRIVE_ROOT / 'class_to_idx_30.json') as f:
    class_to_idx = json.load(f)
NUM_CLASSES = len(class_to_idx)

with open(DRIVE_ROOT / 'loss_weights_30.json') as f:
    loss_weights_payload = json.load(f)
loss_weights = torch.tensor(loss_weights_payload['weights']).float()

print(f"\nClasses: {NUM_CLASSES} (excluded: {loss_weights_payload['excluded_crops']})")
assert set(loss_weights_payload['crop_order']) == set(class_to_idx.keys()), \
    "MISMATCH: loss weight crop order doesn't match class_to_idx — do not proceed"
assert list(class_to_idx.values()) == list(range(NUM_CLASSES)), \
    "MISMATCH: class_to_idx indices are not contiguous 0..N-1 — do not proceed"
print("✅ class_to_idx and loss_weights verified consistent")

# --- Sanity check everything needed is present before writing any training code ---
required = {
    'train_manifest_30.csv': DRIVE_ROOT / 'train_manifest_30.csv',
    'val_manifest_30.csv':   DRIVE_ROOT / 'val_manifest_30.csv',
    'test_manifest_30.csv':  DRIVE_ROOT / 'test_manifest_30.csv',
    'class_to_idx_30.json':  DRIVE_ROOT / 'class_to_idx_30.json',
    'loss_weights_30.json':  DRIVE_ROOT / 'loss_weights_30.json',
    'DATA_ROOT populated':   DATA_ROOT,
    'backgrounds populated': BG_POOL_DIR,
    'OOD_ROOT':              OOD_ROOT,
}
all_ok = True
for name, path in required.items():
    exists = path.exists() and (path.is_file() or any(path.iterdir()))
    print(f"  {'✅' if exists else '❌ MISSING'}  {name}")
    all_ok = all_ok and exists

print(f"\nGPU available: {torch.cuda.is_available()}",
      f"({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "")
print("\n✅ Section A complete — safe to proceed to Section B" if all_ok else "\n❌ Fix missing items before proceeding")

Mounted at /content/drive
Extracting resized_256 from Drive parts...
✅ Image data extracted
Restoring backgrounds from Drive...
✅ Restored 1,400 background images

Classes: 30 (excluded: ['cotton', 'soybean'])
✅ class_to_idx and loss_weights verified consistent
  ✅  train_manifest_30.csv
  ✅  val_manifest_30.csv
  ✅  test_manifest_30.csv
  ✅  class_to_idx_30.json
  ✅  loss_weights_30.json
  ✅  DATA_ROOT populated
  ✅  backgrounds populated
  ✅  OOD_ROOT

GPU available: True (Tesla T4)

✅ Section A complete — safe to proceed to Section B


# SECTION B: CropDataset — 30-class, bg_type-routed transforms


In [ ]:
import cv2
import numpy as np
import random
import pandas as pd
from pathlib import Path
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset

NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]

class RandomBackgroundPaste(A.ImageOnlyTransform):
    def __init__(self, bg_pool_dir, p=0.5):
        super().__init__(p=p)
        self.bg_paths = list(Path(bg_pool_dir).glob('*'))
        assert len(self.bg_paths) > 0, f"No backgrounds found in {bg_pool_dir}"

    def apply(self, img, **kwargs):
        h, w = img.shape[:2]
        bg_path = random.choice(self.bg_paths)
        bg = np.array(Image.open(bg_path).convert('RGB').resize((w, h)))
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        _, otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        border = np.concatenate([
            otsu[:5, :].flatten(), otsu[-5:, :].flatten(),
            otsu[:, :5].flatten(), otsu[:, -5:].flatten()
        ])
        bg_is_white = (border == 255).mean() > 0.5
        leaf_mask = (otsu == 0) if bg_is_white else (otsu == 255)
        leaf_mask3 = np.stack([leaf_mask] * 3, axis=-1)
        return np.where(leaf_mask3, img, bg).astype(np.uint8)

def build_transforms(img_size, bg_pool_dir):
    paste_eligible_transform = A.Compose([
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.6, 1.0), interpolation=cv2.INTER_CUBIC),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Rotate(limit=30, p=0.5, border_mode=cv2.BORDER_REFLECT),
        A.Transpose(p=0.3),
        RandomBackgroundPaste(bg_pool_dir=bg_pool_dir, p=0.5),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25),
            A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=20),
        ], p=0.6),
        A.GaussianBlur(blur_limit=(3, 5), p=0.15),
        A.CoarseDropout(num_holes_range=(1, 6), hole_height_range=(8, 24), hole_width_range=(8, 24), p=0.3),
        A.Normalize(mean=NORM_MEAN, std=NORM_STD),
        ToTensorV2(),
    ])
    natural_transform = A.Compose([
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.7, 1.0), interpolation=cv2.INTER_CUBIC),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Rotate(limit=20, p=0.4, border_mode=cv2.BORDER_REFLECT),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3),
            A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=35, val_shift_limit=25),
            A.RGBShift(r_shift_limit=15, g_shift_limit=15, b_shift_limit=15),
        ], p=0.7),
        A.OneOf([
            A.ImageCompression(quality_range=(40, 95)),
            A.Downscale(scale_range=(0.4, 0.8)),
        ], p=0.3),
        A.GaussianBlur(blur_limit=(3, 7), p=0.2),
        A.GaussNoise(p=0.2),
        A.CoarseDropout(num_holes_range=(1, 6), hole_height_range=(8, 24), hole_width_range=(8, 24), p=0.3),
        A.Normalize(mean=NORM_MEAN, std=NORM_STD),
        ToTensorV2(),
    ])
    val_transform = A.Compose([
        A.SmallestMaxSize(max_size=IMG_SIZE, interpolation=cv2.INTER_CUBIC),
        A.CenterCrop(height=IMG_SIZE, width=IMG_SIZE, pad_if_needed=True,
                 border_mode=cv2.BORDER_REFLECT),
        A.Normalize(mean=NORM_MEAN, std=NORM_STD),
        ToTensorV2(),
    ])
    return paste_eligible_transform, natural_transform, val_transform

paste_eligible_transform, natural_transform, val_transform = build_transforms(IMG_SIZE, BG_POOL_DIR)


class CropDataset(Dataset):
    """Routes each row to the correct transform based on bg_type.
    Missing files are logged and skipped (returns None; use the
    collate_fn below to drop them) rather than crashing mid-epoch."""

    def __init__(self, manifest_path, class_to_idx, data_root, split='train'):
        self.df = pd.read_csv(manifest_path)
        self.class_to_idx = class_to_idx
        self.data_root = Path(data_root)
        self.split = split  # 'train' | 'val' | 'test'
        self._missing_logged = 0

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fpath = self.data_root / row['filepath']

        if not fpath.exists():
            self._missing_logged += 1
            if self._missing_logged <= 20:  # cap log spam
                print(f"⚠️  [{self.split}] missing file, skipping: {row['filepath']}")
            return None

        try:
            img = np.array(Image.open(fpath).convert('RGB'))
        except Exception as e:
            print(f"⚠️  [{self.split}] unreadable file, skipping: {row['filepath']} ({e})")
            return None

        if self.split == 'train':
            bg_type = row.get('bg_type', 'natural')
            transform = paste_eligible_transform if bg_type == 'paste_eligible' else natural_transform
        else:
            transform = val_transform

        img_tensor = transform(image=img)['image']
        label_idx = self.class_to_idx[row['crop']]
        return img_tensor, label_idx, row['filepath']


def safe_collate(batch):
    """Drops None entries (from missing/corrupt files) before batching."""
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None
    imgs, labels, fpaths = zip(*batch)
    import torch as _torch
    return _torch.stack(imgs), _torch.tensor(labels), list(fpaths)


# --- Build datasets and run a smoke test before trusting anything downstream ---
train_ds = CropDataset(DRIVE_ROOT / 'train_manifest_30.csv', class_to_idx, DATA_ROOT, split='train')
val_ds   = CropDataset(DRIVE_ROOT / 'val_manifest_30.csv',   class_to_idx, DATA_ROOT, split='val')
test_ds  = CropDataset(DRIVE_ROOT / 'test_manifest_30.csv',  class_to_idx, DATA_ROOT, split='test')

print(f"train_ds: {len(train_ds):,}  val_ds: {len(val_ds):,}  test_ds: {len(test_ds):,}")

sample = train_ds[0]
if sample is not None:
    img_t, label, fp = sample
    print(f"Smoke test OK — tensor shape {img_t.shape}, label {label}, file {fp}")
else:
    print("⚠️ First sample was missing — dataset may have file path issues, check DATA_ROOT")

train_ds: 130,347  val_ds: 25,481  test_ds: 24,993
Smoke test OK — tensor shape torch.Size([3, 238, 238]), label 0, file Apple/Apple_scab/48420763-bd18-49b4-9ab4-69319b5733f2___FREC_Scab 3201_new30degFlipLR.jpg


# SECTION C: DataLoaders


In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64  # adjust down if hit OOM on your Colab GPU tier

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True,
    collate_fn=safe_collate,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True,
    collate_fn=safe_collate,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True,
    collate_fn=safe_collate,
)

print(f"train_loader: {len(train_loader)} batches  ({len(train_ds):,} samples)")
print(f"val_loader:   {len(val_loader)} batches  ({len(val_ds):,} samples)")
print(f"test_loader:  {len(test_loader)} batches  ({len(test_ds):,} samples)")

# --- Sanity check: pull one batch, confirm shapes and a reasonable class spread ---
batch = next(iter(train_loader))
if batch is not None:
    imgs, labels, fpaths = batch
    print(f"\nBatch shape: {imgs.shape}, labels shape: {labels.shape}")
    print(f"Label range: {labels.min().item()}–{labels.max().item()} (expect 0–{NUM_CLASSES-1})")
    from collections import Counter
    idx_to_crop = {v: k for k, v in class_to_idx.items()}
    dist = Counter(idx_to_crop[l.item()] for l in labels)
    print(f"Classes in this batch: {dict(dist)}")
else:
    print("⚠️ First batch was entirely None — check DATA_ROOT / manifest paths")

train_loader: 2036 batches  (130,347 samples)
val_loader:   399 batches  (25,481 samples)
test_loader:  391 batches  (24,993 samples)

Batch shape: torch.Size([64, 3, 238, 238]), labels shape: torch.Size([64])
Label range: 0–28 (expect 0–29)
Classes in this batch: {'peach': 2, 'tea': 24, 'squash': 1, 'coconut': 4, 'corn': 2, 'tomato': 8, 'apple': 4, 'coffee': 2, 'bell_pepper': 2, 'strawberry': 2, 'grape': 4, 'groundnut': 1, 'paddy': 2, 'blueberry': 1, 'potato': 2, 'pineapple': 2, 'chilli': 1}


# SECTION D: DINOv2 ViT-S/14 with staged fine-tuning + resume


In [ ]:

import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class DinoV2Classifier(nn.Module):
    def __init__(self, num_classes, backbone_name='dinov2_vits14'):
        super().__init__()
        self.backbone = torch.hub.load('facebookresearch/dinov2', backbone_name)
        self.head = nn.Linear(self.backbone.embed_dim, num_classes)  # 384 for ViT-S

    def forward(self, x):
        feats = self.backbone(x)   # [B, 384] — CLS token pooled output
        return self.head(feats)


def set_trainable_stage(model, stage: str):
    """stage in {'A', 'B', 'C'}. Controls what's frozen/unfrozen.
    Returns param groups for the optimizer (differential LR)."""
    for p in model.backbone.parameters():
        p.requires_grad = False
    for p in model.head.parameters():
        p.requires_grad = True

    if stage == 'A':
        # Linear probe — backbone fully frozen
        param_groups = [{'params': model.head.parameters(), 'lr': 1e-3}]

    elif stage == 'B':
        # Unfreeze last 4 transformer blocks + head
        last_blocks = model.backbone.blocks[-4:]
        for block in last_blocks:
            for p in block.parameters():
                p.requires_grad = True
        param_groups = [
            {'params': [p for b in last_blocks for p in b.parameters()], 'lr': 1e-5},
            {'params': model.head.parameters(), 'lr': 1e-4},
        ]

    elif stage == 'C':
        # Full unfreeze
        for p in model.backbone.parameters():
            p.requires_grad = True
        param_groups = [{'params': model.parameters(), 'lr': 5e-6}]

    else:
        raise ValueError(f"Unknown stage: {stage}")

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"Stage {stage}: {n_trainable:,} / {n_total:,} params trainable")
    return param_groups


model = DinoV2Classifier(NUM_CLASSES).to(device)
print(f"\nModel loaded — head output: {NUM_CLASSES} classes, embed_dim: {model.backbone.embed_dim}")

# Resume logic — checkpoint holds EVERYTHING needed to continue:
# model state, optimizer state, scheduler state, epoch, stage, best F1.
# This cell is idempotent: safe to re-run after any disconnect.

CKPT_PATH = CKPT_DIR / 'latest_checkpoint.pt'
BEST_CKPT_PATH = CKPT_DIR / 'best_checkpoint.pt'

def load_checkpoint_if_exists():
    if CKPT_PATH.exists():
        print(f"Found checkpoint at {CKPT_PATH} — resuming")
        ckpt = torch.load(CKPT_PATH, map_location=device)
        model.load_state_dict(ckpt['model_state'])
        assert ckpt['class_to_idx'] == class_to_idx, \
            "❌ Checkpoint class_to_idx doesn't match current 30-class mapping — do not resume blindly"
        print(f"  Resumed at epoch {ckpt['epoch']}, stage {ckpt['stage']}, best F1 {ckpt['best_f1']:.4f}")
        return ckpt
    else:
        print("No checkpoint found — starting fresh from Stage A, epoch 1")
        return None

resume_state = load_checkpoint_if_exists()
current_stage = resume_state['stage'] if resume_state else 'A'
start_epoch   = resume_state['epoch'] + 1 if resume_state else 1
best_f1       = resume_state['best_f1'] if resume_state else 0.0

param_groups = set_trainable_stage(model, current_stage)

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 201MB/s]



Model loaded — head output: 30 classes, embed_dim: 384
Found checkpoint at /content/drive/MyDrive/CropClassifier_v2/models/stage1_dinov2_30class/latest_checkpoint.pt — resuming
  Resumed at epoch 15, stage B, best F1 0.9998
Stage B: 7,112,478 / 22,068,126 params trainable


# SECTION E: Loss, Optimizer, Scheduler

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

criterion = nn.CrossEntropyLoss(weight=loss_weights.to(device), label_smoothing=0.1)

STAGE_EPOCHS = {'A': 5, 'B': 15, 'C': 25}  # cumulative epoch counts at which each stage ends
STAGE_T_MAX  = {'A': 5, 'B': 10, 'C': 10}  # per-stage cosine cycle length

optimizer = AdamW(param_groups, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=STAGE_T_MAX[current_stage])

# If resuming mid-stage, restore optimizer/scheduler state too —
# otherwise momentum/LR schedule would reset incorrectly on resume
if resume_state is not None:
    try:
        optimizer.load_state_dict(resume_state['optimizer_state'])
        scheduler.load_state_dict(resume_state['scheduler_state'])
        print("✅ Optimizer + scheduler state restored")
    except Exception as e:
        print(f"⚠️ Could not restore optimizer/scheduler state ({e}) — continuing with fresh state, "
              f"model weights are still correctly restored")

print(f"\nReady to train — starting at epoch {start_epoch}, stage {current_stage}")
print(f"Loss weights: min={loss_weights.min():.3f}, max={loss_weights.max():.3f}")

✅ Optimizer + scheduler state restored

Ready to train — starting at epoch 12, stage B
Loss weights: min=0.201, max=2.471


# SECTION F: Training loop — per-epoch checkpointing, resumable


In [ ]:
import json, time
from datetime import datetime
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

F1_HISTORY_PATH = CKPT_DIR / 'f1_history.json'
WATCH_LIST = ['wheat', 'snake_gourd', 'onion', 'dragon_fruit', 'jute', 'blackgram', 'tea']
# (cotton, soybean dropped from watch list — excluded from this 30-class run)
F1_FLOOR = 0.5
idx_to_crop = {v: k for k, v in class_to_idx.items()}

def load_f1_history():
    if F1_HISTORY_PATH.exists():
        with open(F1_HISTORY_PATH) as f:
            return json.load(f)
    return []

def save_f1_history(history):
    """Atomic write — never leaves a half-written history file on Drive."""
    tmp = F1_HISTORY_PATH.with_suffix('.json.tmp')
    with open(tmp, 'w') as f:
        json.dump(history, f, indent=2)
    tmp.replace(F1_HISTORY_PATH)

def save_checkpoint(path, epoch, stage, best_f1):
    """Atomic write for model checkpoints too."""
    tmp = path.with_suffix('.pt.tmp')
    torch.save({
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'epoch': epoch,
        'stage': stage,
        'best_f1': best_f1,
        'class_to_idx': class_to_idx,
    }, tmp)
    tmp.replace(path)


def train_one_epoch(loader):
    model.train()
    total_loss, n_batches = 0.0, 0
    for batch in tqdm(loader, desc="train", leave=False):
        if batch is None:
            continue
        imgs, labels, _ = batch
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)


@torch.no_grad()
def validate(loader):
    model.eval()
    total_loss, n_batches = 0.0, 0
    all_preds, all_labels = [], []
    for batch in tqdm(loader, desc="val", leave=False):
        if batch is None:
            continue
        imgs, labels, _ = batch
        imgs, labels = imgs.to(device), labels.to(device)

        outputs = model(imgs)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        n_batches += 1

        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    val_loss = total_loss / max(n_batches, 1)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    per_class_f1 = f1_score(all_labels, all_preds, average=None, zero_division=0,
                             labels=list(range(NUM_CLASSES)))
    per_class_f1_dict = {idx_to_crop[i]: float(f) for i, f in enumerate(per_class_f1)}
    return val_loss, macro_f1, per_class_f1_dict


def print_per_class_table(per_class_f1_dict):
    print(f"\n{'Crop':20s} {'F1':>6}")
    print("-" * 28)
    for crop, f1 in sorted(per_class_f1_dict.items(), key=lambda x: x[1]):
        flag = "  ⚠️ BELOW FLOOR" if f1 < F1_FLOOR else ""
        watch = "  [watch]" if crop in WATCH_LIST else ""
        print(f"  {crop:20s} {f1:>6.3f}{watch}{flag}")



# Main loop — reads current_stage / start_epoch / best_f1 from
# Section D's resume logic. Safe to re-run after any disconnect.

history = load_f1_history()
epochs_since_improve = 0
MAX_EPOCH = STAGE_EPOCHS['C']  # 25 — hard ceiling, Stage C is optional/triggered manually

epoch = start_epoch
while epoch <= MAX_EPOCH:
    # --- stage transition check ---
    if epoch > STAGE_EPOCHS[current_stage]:
        if current_stage == 'A':
            current_stage = 'B'
        elif current_stage == 'B':
            print("\n🔔 Stage B complete. Stage C (full unfreeze) is optional — "
                  "only trigger manually if OOD plateaus. Stopping here for now.")
            break
        param_groups = set_trainable_stage(model, current_stage)
        optimizer = AdamW(param_groups, weight_decay=1e-4)
        scheduler = CosineAnnealingLR(optimizer, T_max=STAGE_T_MAX[current_stage])
        criterion = nn.CrossEntropyLoss(weight=loss_weights.to(device), label_smoothing=0.1)
        epochs_since_improve = 0
        print(f"\n{'='*50}\nTransitioned to Stage {current_stage} at epoch {epoch}\n{'='*50}")

    print(f"\n--- Epoch {epoch} (Stage {current_stage}) ---")
    t0 = time.time()

    train_loss = train_one_epoch(train_loader)
    val_loss, macro_f1, per_class_f1 = validate(val_loader)
    scheduler.step()

    elapsed = time.time() - t0
    print(f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
          f"macro_f1={macro_f1:.4f}  ({elapsed:.0f}s)")
    print_per_class_table(per_class_f1)

    # --- persist history EVERY epoch, not just at the end ---
    history.append({
        'epoch': epoch, 'stage': current_stage,
        'train_loss': train_loss, 'val_loss': val_loss,
        'macro_f1': macro_f1, 'per_class_f1': per_class_f1,
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M'),
    })
    save_f1_history(history)

    # --- checkpointing ---
    save_checkpoint(CKPT_PATH, epoch, current_stage, best_f1)  # latest, every epoch — for resume
    if macro_f1 > best_f1:
        best_f1 = macro_f1
        save_checkpoint(BEST_CKPT_PATH, epoch, current_stage, best_f1)
        epochs_since_improve = 0
        print(f"  ✅ New best macro F1: {best_f1:.4f} — saved to best_checkpoint.pt")
    else:
        epochs_since_improve += 1
        print(f"  No improvement ({epochs_since_improve} epoch(s) since best={best_f1:.4f})")

    # --- early stopping, per stage (does not carry across stages) ---
    if epochs_since_improve >= 5:
        print(f"\n🛑 Early stopping — no improvement in 5 epochs within Stage {current_stage}")
        if current_stage == 'A':
            current_stage = 'B'
            param_groups = set_trainable_stage(model, current_stage)
            optimizer = AdamW(param_groups, weight_decay=1e-4)
            scheduler = CosineAnnealingLR(optimizer, T_max=STAGE_T_MAX[current_stage])
            epochs_since_improve = 0
            epoch += 1
            continue
        else:
            print("Stopping training.")
            break

    epoch += 1

print(f"\n✅ Training loop finished at epoch {epoch-1}, stage {current_stage}, best F1 {best_f1:.4f}")
print(f"   f1_history.json: {len(history)} epochs recorded on Drive")


--- Epoch 12 (Stage B) ---


train:   0%|          | 0/2036 [00:00<?, ?it/s]

val:   0%|          | 0/399 [00:00<?, ?it/s]

train_loss=1.1371  val_loss=1.4944  macro_f1=0.9997  (1190s)

Crop                     F1
----------------------------
  pineapple             0.996
  mango                 0.998
  sugarcane             0.999
  cherry                0.999
  potato                1.000
  apple                 1.000
  bean                  1.000
  bell_pepper           1.000
  blackgram             1.000  [watch]
  blueberry             1.000
  chilli                1.000
  coconut               1.000
  coffee                1.000
  corn                  1.000
  dragon_fruit          1.000  [watch]
  grape                 1.000
  groundnut             1.000
  jute                  1.000  [watch]
  lemon                 1.000
  onion                 1.000  [watch]
  orange                1.000
  paddy                 1.000
  peach                 1.000
  raspberry             1.000
  snake_gourd           1.000  [watch]
  squash                1.000
  strawberry            1.000
  tea                   1.

train:   0%|          | 0/2036 [00:00<?, ?it/s]

...

val:   0%|          | 0/399 [00:00<?, ?it/s]

train_loss=1.1307  val_loss=1.4926  macro_f1=0.9997  (1196s)

Crop                     F1
----------------------------
  pineapple             0.996
  mango                 0.998
  sugarcane             0.999
  cherry                0.999
  potato                1.000
  apple                 1.000
  bean                  1.000
  bell_pepper           1.000
  blackgram             1.000  [watch]
  blueberry             1.000
  chilli                1.000
  coconut               1.000
  coffee                1.000
  corn                  1.000
  dragon_fruit          1.000  [watch]
  grape                 1.000
  groundnut             1.000
  jute                  1.000  [watch]
  lemon                 1.000
  onion                 1.000  [watch]
  orange                1.000
  paddy                 1.000
  peach                 1.000
  raspberry             1.000
  snake_gourd           1.000  [watch]
  squash                1.000
  strawberry            1.000
  tea                   1.

train:   0%|          | 0/2036 [00:00<?, ?it/s]

val:   0%|          | 0/399 [00:00<?, ?it/s]

train_loss=1.1294  val_loss=1.4926  macro_f1=0.9997  (1194s)

Crop                     F1
----------------------------
  pineapple             0.996
  mango                 0.998
  sugarcane             0.999
  cherry                0.999
  potato                1.000
  apple                 1.000
  bean                  1.000
  bell_pepper           1.000
  blackgram             1.000  [watch]
  blueberry             1.000
  chilli                1.000
  coconut               1.000
  coffee                1.000
  corn                  1.000
  dragon_fruit          1.000  [watch]
  grape                 1.000
  groundnut             1.000
  jute                  1.000  [watch]
  lemon                 1.000
  onion                 1.000  [watch]
  orange                1.000
  paddy                 1.000
  peach                 1.000
  raspberry             1.000
  snake_gourd           1.000  [watch]
  squash                1.000
  strawberry            1.000
  tea                   1.

In [ ]:
import torch
from pathlib import Path
from PIL import Image
import numpy as np
from sklearn.metrics import confusion_matrix
import json
from datetime import datetime


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CKPT_PATH = CKPT_DIR / 'latest_checkpoint.pt'
BEST_CKPT_PATH = CKPT_DIR / 'best_checkpoint.pt'
OOD_RESULTS_PATH = CKPT_DIR / 'ood_results.json'
EXCLUDE_CROPS = ['cotton', 'soybean']  # not in this 30-class run
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}


def load_model_from_checkpoint(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    m = DinoV2Classifier(NUM_CLASSES).to(device)
    m.load_state_dict(ckpt['model_state'])
    m.eval()
    assert ckpt['class_to_idx'] == class_to_idx, "class_to_idx mismatch — do not evaluate blindly"
    return m, ckpt['epoch'], ckpt['stage']

# Build a case-insensitive lookup: lowercased folder name -> real class_to_idx key
crop_lookup = {k.lower(): k for k in class_to_idx.keys()}

@torch.no_grad()
def run_ood_eval(model, ood_root=OOD_ROOT):
    ood_root = Path(ood_root)
    all_folders = sorted(p for p in ood_root.iterdir() if p.is_dir())

    all_preds, all_labels, all_crops = [], [], []
    per_crop_correct, per_crop_total = {}, {}
    matched_folders = 0

    for crop_folder in all_folders:
        folder_key = crop_folder.name.lower()
        if folder_key in EXCLUDE_CROPS:
            continue
        if folder_key not in crop_lookup:
            print(f"  ⚠️ skipping {crop_folder.name} — no match in 30-class mapping")
            continue

        crop_name = crop_lookup[folder_key]   # canonical lowercase key used by the model
        matched_folders += 1
        true_idx = class_to_idx[crop_name]
        imgs = [p for p in crop_folder.iterdir() if p.suffix.lower() in IMG_EXTS]
        per_crop_correct[crop_name] = 0
        per_crop_total[crop_name] = 0

        for img_path in imgs:
            try:
                img = np.array(Image.open(img_path).convert('RGB'))
            except Exception:
                continue
            img_t = val_transform(image=img)['image'].unsqueeze(0).to(device)
            pred = model(img_t).argmax(dim=1).item()

            all_preds.append(pred)
            all_labels.append(true_idx)
            all_crops.append(crop_name)
            per_crop_total[crop_name] += 1
            if pred == true_idx:
                per_crop_correct[crop_name] += 1

    # Hard guard: if most crops failed to match, refuse to report a misleading number
    expected_min = NUM_CLASSES - len(EXCLUDE_CROPS) - 5  # allow a little slack
    if matched_folders < expected_min:
        raise RuntimeError(
            f"Only matched {matched_folders}/{NUM_CLASSES - len(EXCLUDE_CROPS)} expected crops — "
            f"folder naming likely doesn't align with class_to_idx. Fix before trusting any OOD number."
        )

    macro_acc = np.mean([per_crop_correct[c] / max(per_crop_total[c], 1) for c in per_crop_total])
    per_crop_acc = {c: per_crop_correct[c] / max(per_crop_total[c], 1) for c in per_crop_total}

    print(f"\nMatched {matched_folders} crop folders")
    print(f"OOD macro accuracy: {macro_acc:.4f}\n")
    print(f"{'Crop':20s} {'Acc':>6}  {'N':>4}")
    print("-" * 34)
    for crop, acc in sorted(per_crop_acc.items(), key=lambda x: x[1]):
        print(f"  {crop:20s} {acc:>6.3f}  {per_crop_total[crop]:>4d}")

    cm = confusion_matrix(all_labels, all_preds, labels=list(range(NUM_CLASSES)))
    confused_pairs = []
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            if i != j and cm[i, j] > 0:
                confused_pairs.append((idx_to_crop[i], idx_to_crop[j], int(cm[i, j])))
    confused_pairs.sort(key=lambda x: -x[2])
    print(f"\nTop confused pairs (true -> predicted):")
    for true_c, pred_c, count in confused_pairs[:10]:
        print(f"  {true_c} -> {pred_c}: {count}")

    return macro_acc, per_crop_acc, confused_pairs


model_eval, ckpt_epoch, ckpt_stage = load_model_from_checkpoint(BEST_CKPT_PATH)
print(f"Evaluating checkpoint from epoch {ckpt_epoch}, stage {ckpt_stage}\n")
macro_acc, per_crop_acc, confused_pairs = run_ood_eval(model_eval)

results_history = []
if OOD_RESULTS_PATH.exists():
    with open(OOD_RESULTS_PATH) as f:
        results_history = json.load(f)
results_history.append({
    'epoch': ckpt_epoch, 'stage': ckpt_stage,
    'macro_acc': macro_acc, 'per_crop_acc': per_crop_acc,
    'top_confused_pairs': confused_pairs[:10],
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M'),
})
tmp = OOD_RESULTS_PATH.with_suffix('.json.tmp')
with open(tmp, 'w') as f:
    json.dump(results_history, f, indent=2)
tmp.replace(OOD_RESULTS_PATH)
print(f"\n✅ Saved to {OOD_RESULTS_PATH}")

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


Evaluating checkpoint from epoch 9, stage B


Matched 30 crop folders
OOD macro accuracy: 0.4548

Crop                    Acc     N
----------------------------------
  cherry                0.000    11
  bean                  0.000    11
  bell_pepper           0.000     0
  peach                 0.000    10
  squash                0.000     9
  tea                   0.000    11
  lemon                 0.083    12
  orange                0.111     9
  potato                0.118    17
  blueberry             0.167     6
  mango                 0.214    14
  raspberry             0.286    14
  apple                 0.300    10
  tomato                0.412    17
  grape                 0.417    12
  onion                 0.429     7
  pineapple             0.500     8
  coffee                0.556     9
  jute                  0.556     9
  corn                  0.571    14
  chilli                0.750    12
  sugarcane             0.778     9
  groundnut             0.818    11
  str

In [ ]:
bp_folder = OOD_ROOT / 'Bell_Pepper'  # check actual casing/spelling on disk
print(list(OOD_ROOT.glob('*ell*epper*')))  # find the real folder name

[PosixPath('/content/drive/MyDrive/disease_detection/unseen/Diseased_and_healthy/bell_pepper')]


In [ ]:
bp_folder = OOD_ROOT / 'bell_pepper'
all_files = list(bp_folder.iterdir())
print(f"Total items in folder: {len(all_files)}")
for f in all_files[:20]:
    print(f"  {f.name}  suffix={f.suffix!r}  is_file={f.is_file()}")

matched = [p for p in all_files if p.suffix.lower() in IMG_EXTS]
print(f"\nMatched IMG_EXTS: {len(matched)}")

Total items in folder: 0

Matched IMG_EXTS: 0


# Standalone Inference

In [ ]:
import sys
import json
from pathlib import Path

import torch
import torch.nn as nn
import numpy as np
import cv2
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Config — adjust paths if running outside the original Drive layout
DRIVE_ROOT = Path('/content/drive/MyDrive/CropClassifier_v2')
CKPT_PATH = DRIVE_ROOT / 'models' / 'stage1_dinov2_30class' / 'best_checkpoint.pt'
IMG_SIZE = 238
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model definition (must match training exactly)
class DinoV2Classifier(nn.Module):
    def __init__(self, num_classes, backbone_name='dinov2_vits14'):
        super().__init__()
        self.backbone = torch.hub.load('facebookresearch/dinov2', backbone_name)
        self.head = nn.Linear(self.backbone.embed_dim, num_classes)

    def forward(self, x):
        feats = self.backbone(x)
        return self.head(feats)

# Same eval transform used for val/OOD — resize-then-crop, robust to any input size
infer_transform = A.Compose([
    A.SmallestMaxSize(max_size=IMG_SIZE, interpolation=cv2.INTER_CUBIC),
    A.CenterCrop(height=IMG_SIZE, width=IMG_SIZE, pad_if_needed=True,
                 border_mode=cv2.BORDER_REFLECT),
    A.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ToTensorV2(),
])


def load_model(ckpt_path=CKPT_PATH):
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    class_to_idx = ckpt['class_to_idx']
    idx_to_class = {v: k for k, v in class_to_idx.items()}

    model = DinoV2Classifier(num_classes=len(class_to_idx)).to(DEVICE)
    model.load_state_dict(ckpt['model_state'])
    model.eval()

    print(f"Loaded checkpoint — epoch {ckpt['epoch']}, stage {ckpt['stage']}, "
          f"best_f1 {ckpt['best_f1']:.4f}, {len(class_to_idx)} classes")
    return model, idx_to_class


@torch.no_grad()
def predict(image_path, model, idx_to_class, top_k=3):
    img = np.array(Image.open(image_path).convert('RGB'))
    img_t = infer_transform(image=img)['image'].unsqueeze(0).to(DEVICE)

    logits = model(img_t)
    probs = torch.softmax(logits, dim=1)[0]

    top_probs, top_idxs = probs.topk(top_k)
    results = [
        {'crop': idx_to_class[idx.item()], 'confidence': round(prob.item(), 4)}
        for prob, idx in zip(top_probs, top_idxs)
    ]
    return results




In [ ]:
model, idx_to_class = load_model()
results = predict('/content/dragon_fruit.jpg', model, idx_to_class)
for r in results:
    print(r)

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


Loaded checkpoint — epoch 9, stage B, best_f1 0.9998, 30 classes
{'crop': 'dragon_fruit', 'confidence': 0.5827}
{'crop': 'pineapple', 'confidence': 0.1296}
{'crop': 'onion', 'confidence': 0.0448}
